# ChildSafe — detector vật nguy cơ mở rộng
Upload ZIP từ package_hazard_dataset.py thành một Kaggle Dataset riêng, rồi thêm vào notebook.
Chỉ các lớp có nhãn thật ở cả train/val/test mới được train. Không tự sinh nhãn từ model.
Bật GPU. Internet chỉ cần để cài dependency. Backbone phải có trong gói local.
Ưu tiên segmentation để đo vùng vật; bounding box chỉ dùng cảnh giác nghi ngờ.
Kết quả ảnh tĩnh không chứng minh hiệu quả camera, kích thước vật lý hay nhận diện nghẹn.


In [ ]:
%pip install -q ultralytics==8.3.253


In [ ]:
from pathlib import Path
import json, yaml, torch
assert torch.cuda.is_available(), 'Enable GPU in Kaggle settings'
candidates=[p for p in Path('/kaggle/input').rglob('dataset.yaml') if (p.parent/'report.json').is_file()]
assert len(candidates)==1, f'Expected one reviewed object dataset, found {candidates}'
source=candidates[0].parent
report=json.loads((source/'report.json').read_text())
backbones=list(Path('/kaggle/input').rglob('yolo11n-seg.pt'))
assert len(backbones)==1, 'Package one explicit local backbone with the dataset'
work=Path('/kaggle/working/hazard_data'); work.mkdir(exist_ok=False)
cfg=yaml.safe_load((source/'dataset.yaml').read_text()); cfg['path']=str(source)
(work/'dataset.yaml').write_text(yaml.safe_dump(cfg))
import shutil
for name in ('manifest.csv','report.json'): shutil.copy2(source/name, work/name)
print(report)


In [ ]:
"""Train the classes ACTUALLY annotated; keep person detection separate at runtime."""
import argparse
import hashlib
import json
from pathlib import Path

import yaml


def train(data, output, task='segment', epochs=60, size=960, batch=8, device='0',
          backbone=Path('models/yolo11n-seg.pt')):
    data,output,backbone = Path(data),Path(output),Path(backbone)
    if output.exists():
        raise FileExistsError('use a new output directory')
    if not backbone.is_file():
        raise FileNotFoundError('explicit local backbone required; never download implicitly')
    report = json.loads((data.parent/'report.json').read_text(encoding='utf-8'))
    if report['task'] != task:
        raise ValueError('dataset annotation task does not match training task')
    names = yaml.safe_load(data.read_text(encoding='utf-8'))['names']
    if list(names.values()) != report['classes']:
        raise ValueError('dataset class inventory differs from reviewed report')
    from ultralytics import YOLO
    architecture = 'yolo11n-seg.yaml' if task == 'segment' else 'yolo11n.yaml'
    model = YOLO(architecture).load(str(backbone))
    # Val controls model selection; test is used once after training, never for tuning.
    model.train(data=str(data.resolve()),epochs=epochs,imgsz=size,batch=batch,device=device,
                workers=2,project=str(output.parent),name=output.name,exist_ok=False,
                seed=42,deterministic=True,patience=15,amp=False,plots=True,
                cache=False,close_mosaic=10,save=True)
    best = Path(model.trainer.best); evaluated = YOLO(str(best))
    result = evaluated.val(data=str(data.resolve()),split='test',device=device,imgsz=size,
                           plots=True,project=str(output),name='heldout_test')
    metrics = result.seg if task == 'segment' else result.box
    payload = dict(task=task,classes=evaluated.names,checkpoint=best.name,
        checkpoint_sha256=hashlib.sha256(best.read_bytes()).hexdigest(),
        backbone_sha256=hashlib.sha256(backbone.read_bytes()).hexdigest(),
        dataset_manifest_sha256=hashlib.sha256((data.parent/'manifest.csv').read_bytes()).hexdigest(),
        test_metrics=result.results_dict,
        per_class={evaluated.names[int(cid)]:dict(precision=float(metrics.p[i]),recall=float(metrics.r[i]),
                   map50=float(metrics.ap50[i]),map50_95=float(metrics.ap[i]))
                   for i,cid in enumerate(metrics.ap_class_index)},
        limitation='Independent camera trials still required. Boxes only permit candidate proximity; no choking diagnosis.')
    (output/'evaluation.json').write_text(json.dumps(payload,indent=2),encoding='utf-8')
    print(json.dumps(payload,indent=2))




In [ ]:
train(work/'dataset.yaml', Path('/kaggle/working/outputs/hazard_detector'),
      task=report['task'], epochs=60, size=960, batch=8, device='0', backbone=backbones[0])


In [ ]:
import zipfile
with zipfile.ZipFile('/kaggle/working/childsafe_hazard_detector_results.zip','x',zipfile.ZIP_DEFLATED) as archive:
    root=Path('/kaggle/working/outputs/hazard_detector')
    for path in root.rglob('*'):
        if path.is_file(): archive.write(path,'hazard_detector/'+path.relative_to(root).as_posix())
print('Download childsafe_hazard_detector_results.zip; evaluate on independent camera sessions before deployment.')
